# 🏥 SyftBox — Data Owner Tutorial

Welcome! This notebook walks you through using **SyftBox** as a **Data Owner** (DO).

## What is a Data Owner?

You hold private data that other people — **Data Scientists** (DS) — want to analyze. SyftBox lets them submit Python jobs that run *on your machine*, against your private data, returning only the results you choose to share back.

You stay in control:
- 🔒 The raw data never leaves your machine.
- ✅ You decide who can connect (peer approval).
- 📋 You review every job's source code before it runs (or whitelist patterns for auto-approval).
- 📤 Only the files in `outputs/` come back to the Data Scientist.

## What's in this tutorial

| Step | What you'll do |
|------|----------------|
| 1 | Install `syft-client` |
| 2 | Configure your DO email |
| 3 | One-time: create your SyftBox token |
| 4 | Log in as Data Owner |
| 5 | Connect with your Data Scientists |
| 6 | Create datasets — both a simple CSV and a CSV-with-PDFs example |
| 7 | List & inspect submitted jobs |
| 8 | Review and approve (or reject) a job |
| 9 | Run approved jobs and share results |
| 10 | Run unattended with `syft_bg` background services |
| 11 | Set up auto-approval for trusted job patterns |
| 12 | Inspect service logs |
| 13 | Reset (only if needed) |

## Before you start

You need:
1. **A Google account** — used as your SyftBox identity. Drive is the transport channel.
2. **OAuth client credentials** (`credentials.json` from Google Cloud Console). One-time setup; see Step 3.
3. **The Data Scientist's email** — they'll send you a peer request, and you decide whether to approve it.

> 💡 This is a **Jupyter notebook running on your local machine** (laptop, desktop, or always-on server). Data Owners need a persistent host because the background services in Step 11 keep running between sessions. If you're running Jupyter on a *headless* remote server (no browser/display), see the heads-up in Step 3 about the OAuth flow.


---
## Step 1 — Install `syft-client`

We pin a specific version so this tutorial keeps working as the library evolves. `reportlab` is only needed for the PDF generation in Step 6b.

In [42]:
%pip install -q "syft-client==0.1.113" reportlab
print("✅ syft-client and reportlab installed.")



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ syft-client and reportlab installed.


---
## Step 2 — Configure

Set your Data Owner email and pre-list the Data Scientists you intend to share datasets with.

In [43]:
import syft_client as sc
print(f"✅ syft-client {sc.__version__} loaded.")


✅ syft-client 0.1.113 loaded.


In [44]:
# ✏️ EDIT THESE
DO_EMAIL  = "ronnie@openmined.org"          # ✏️ your SyftBox/Google email
DS_EMAILS = ["falcon.ronnie@gmail.com"]    # ✏️ Data Scientists you intend to share with

# ── Validation ──────────────────────────────────────────────────────────
issues = []
if DO_EMAIL == "data.owner@example.com" or "@" not in DO_EMAIL:
    issues.append(f"❌  DO_EMAIL ({DO_EMAIL!r}) is still the placeholder or invalid.")
for ds in DS_EMAILS:
    if "@" not in ds or ds == "data.scientist@example.com":
        issues.append(f"❌  DS email ({ds!r}) is still the placeholder or invalid.")

if issues:
    print("⚠️  Fix the following before continuing:\n")
    for msg in issues: print(" ", msg)
else:
    print("✅  Configuration looks good!")
    print(f"    DO email  : {DO_EMAIL}")
    print(f"    DS emails : {DS_EMAILS}")


✅  Configuration looks good!
    DO email  : ronnie@openmined.org
    DS emails : ['falcon.ronnie@gmail.com']


---
## Step 3 — One-time: create your SyftBox token

When SyftBox runs in a Jupyter notebook on your machine, it needs an OAuth token to talk to Google. **One token covers everything** — Drive (transport), Gmail (notifications), and Pub/Sub (email-reply approval).

### What you need

1. A Google Cloud project with the **Drive API**, **Gmail API**, and **Pub/Sub API** enabled.
2. **OAuth 2.0 client credentials** — a `credentials.json` you download from the GCP console.
   - Application type: **Desktop app**
   - Test users: include your DO email
3. Place that file at the path below (or edit `CREDENTIALS_PATH`).

`sc.credentials_to_token(...)` opens a browser, walks you through the OAuth flow, and writes a token JSON you'll reuse forever.

> 💡 **Headless Jupyter (no browser)?** Pass `force_browserless=True` to `credentials_to_token` — it'll print a URL for you to open on a different machine; you paste the resulting code back into the notebook.


In [45]:
from pathlib import Path

# ✏️ EDIT IF NEEDED
CREDENTIALS_PATH = Path("/Users/falcon.ronnie/Downloads/credentials.json").expanduser()   # ✏️ from GCP console
TOKEN_PATH       = Path("~/.syft-creds/token.json").expanduser()          # ✏️ where to save the token

if not CREDENTIALS_PATH.exists():
    print(f"❌  Credentials not found at {CREDENTIALS_PATH}.")
    print("    Download a Desktop OAuth client from Google Cloud Console and save it there.")
elif TOKEN_PATH.exists():
    print(f"✅  Token already exists at {TOKEN_PATH}. Skip the next cell.")
else:
    print(f"📄 Credentials found. Run the next cell to authorize.")


✅  Token already exists at /Users/falcon.ronnie/.syft-creds/token.json. Skip the next cell.


In [46]:
# Run once. On a desktop machine a browser window opens for the OAuth flow;
# on a headless server, set force_browserless=True to copy/paste a URL instead.
# The token is reused on every future run — you won't repeat this.

if not TOKEN_PATH.exists():
    sc.credentials_to_token(
        credentials_path=CREDENTIALS_PATH,
        output_path=TOKEN_PATH,
        do_scopes=True,                    # request the broader DO scope set (Drive + Gmail + Pub/Sub)
        # force_browserless=True,           # ← uncomment if running on a headless server
    )
    print(f"✅  Token written to {TOKEN_PATH}")
else:
    print(f"✅  Token already exists at {TOKEN_PATH} — nothing to do.")


✅  Token already exists at /Users/falcon.ronnie/.syft-creds/token.json — nothing to do.


---
## Step 4 — Log in as Data Owner

`sc.login_do(...)` initializes your Data Owner client using the Drive token from Step 3.

In [47]:
client = sc.login_do(email=DO_EMAIL, token_path=str(TOKEN_PATH))


Ensured directories exist: /Users/falcon.ronnie/SyftBox_ronnie@openmined.org/ronnie@openmined.org/app_data/job
🔑 Logging in as ronnie@openmined.org...
   Environment: Python
Found rolling state with 15 events, applying...
Uploading rolling state with 15 events...

✅ Logged in successfully!
   SyftBox folder : /Users/falcon.ronnie/SyftBox_ronnie@openmined.org
   Version        : 0.1.113

👥 1 peer(s) restored from previous session.
   Run client.sync() to pull the latest updates.


---
## Step 5 — Connect with your Data Scientists

You need an active peer connection with each DS before they can submit jobs. Two ways to set this up:

| Approach | When to use |
|----------|-------------|
| **You add them first** (`client.add_peer(ds_email)`) | You already have the DS's email and want to reach out first. **The default in this tutorial.** |
| **They request, you approve** (`client.approve_peer_request(ds_email)`) | The DS has already sent you a request — it shows up in `client.peers` with state `requested_by_peer`. |

This is **one-time setup per Data Scientist** — once the connection is established, it persists across sessions.


In [48]:
client.peers


PeerList([Peer(email='falcon.ronnie@gmail.com', platforms=[GdriveFilesPlatform(name='Gdrive Files', module_path='google_personal.gdrive_files')], state=<PeerState.ACCEPTED: 'accepted'>, version=VersionInfo(syft_client_version='0.1.113', min_supported_syft_client_version='0.1.93', protocol_version='1.0.0', min_supported_protocol_version='1.0.0', updated_at=datetime.datetime(2026, 5, 7, 22, 18, 32, 74907, tzinfo=TzInfo(UTC))), public_encryption_bundle=None, use_encryption=False)])

### Pre-add the Data Scientists

Iterate over `DS_EMAILS` and add each one. If a peer is already added, this is a no-op.


In [49]:
for ds_email in DS_EMAILS:
    client.add_peer(ds_email)
    print(f"✅ Added {ds_email} as a peer")


ℹ️  Already have a connection with falcon.ronnie@gmail.com (state: accepted).
   Use force=True to resend the request.
✅ Added falcon.ronnie@gmail.com as a peer


---
## Step 6 — Create datasets

Each dataset has **two versions**:

- **Mock**: a synthetic look-alike with the same schema. Data Scientists see this freely and develop their analysis against it.
- **Private**: the real data. Stays on your machine; only results from approved jobs flow back to the DS.

`client.create_dataset(...)` accepts files OR folders for both `mock_path` and `private_path`. Use a folder when your dataset has multiple files (e.g. a CSV index plus companion PDFs).

We'll create two datasets to demonstrate both shapes:

| Dataset | Shape | Used for |
|---------|-------|----------|
| `beach_water_quality` | Single CSV | The example used in the DS tutorial |
| `lab_reports` | CSV index + PDFs | The multi-file pattern (the `load_index_with_companions` helper in the DS tutorial) |

### 6a — Simple dataset: a single CSV

Generates a small water-quality CSV with a mock and private version. The mock uses different RNG seed than the private so the values genuinely differ.

In [50]:
import csv
import random
from datetime import date, timedelta
from pathlib import Path

DATA_DIR = Path("data").resolve()
DATA_DIR.mkdir(exist_ok=True)

def generate_beach_csv(out_path: Path, label: str, seed: int):
    """Generate a beach water quality CSV. Mock and private use different RNG seeds."""
    random.seed(seed)
    stations = [
        ("A1", "Sunset Beach North"),
        ("A2", "Sunset Beach South"),
        ("B1", "Harbor Point"),
        ("B2", "Harbor Cove"),
        ("C1", "Rocky Bay"),
    ]
    rows = []
    for i in range(90):
        station_id, station_name = random.choice(stations)
        d = date(2024, 1, 1) + timedelta(days=i)
        bacteria = max(0, int(random.gauss(120, 50)))
        rows.append({
            "date": d.isoformat(),
            "station_id": station_id,
            "station_name": station_name,
            "temperature_c": round(random.uniform(18, 25), 1),
            "ph": round(random.uniform(7.5, 8.5), 2),
            "salinity_ppt": round(random.uniform(30, 35), 1),
            "bacteria_cfu_ml": bacteria,
            "turbidity_ntu": round(random.uniform(2, 6), 1),
            "dissolved_oxygen_mg_l": round(random.uniform(8, 11), 1),
            "safe_to_swim": bacteria < 100,
            "inspector_id": f"{label}{i:03d}",
            "notes": "",
        })
    with open(out_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

mock_csv    = DATA_DIR / "beach_mock.csv"
private_csv = DATA_DIR / "beach_private.csv"
generate_beach_csv(mock_csv,    label="MOCK", seed=1)
generate_beach_csv(private_csv, label="REAL", seed=42)

print(f"✅  Generated {mock_csv}")
print(f"✅  Generated {private_csv}")


✅  Generated /Users/falcon.ronnie/data/beach_mock.csv
✅  Generated /Users/falcon.ronnie/data/beach_private.csv


In [52]:
# Create the dataset and share it with your Data Scientists.
# `users=DS_EMAILS` grants them read access to the mock + dataset metadata.
# `upload_private=False` (the default) keeps the private file local-only.

dataset = client.create_dataset(
    name="beach_water_quality1",
    mock_path=mock_csv,
    private_path=private_csv,
    summary="Beach water quality readings (90 days, 5 stations)",
    tags=["water", "environmental"],
    users=DS_EMAILS,
)
print(f"✅  Dataset {dataset.name!r} created and shared with {DS_EMAILS}")


✅  Dataset 'beach_water_quality1' created and shared with ['falcon.ronnie@gmail.com']


### 6b — Multi-file dataset: CSV index + PDFs

This is the dataset the DS tutorial's `load_index_with_companions` helper is designed for. The mock and private versions have the **same schema** (so DS code works in both places) but **disjoint content** — the mock uses placeholder names like `Patient MOCK001`, the private uses real-looking names. That keeps the privacy story crisp: a learner looking at `dataset.mock_files` *knows* they're seeing fake data.

> 💡 In a real deployment, your private data would already exist on disk. Here we synthesize it for demonstration.

In [53]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

# Real-looking patient names (used only for the private version)
REAL_PATIENTS = [
    ("Alice Johnson", 34, "F"), ("Bob Smith", 52, "M"),
    ("Carol Davis", 28, "F"), ("David Wilson", 67, "M"),
    ("Eve Martinez", 45, "F"), ("Frank Lee", 39, "M"),
    ("Grace Chen", 31, "F"), ("Henry Park", 58, "M"),
    ("Iris Patel", 42, "F"), ("Jack Brown", 29, "M"),
]

def generate_lab_pdf(out_path: Path, name: str, age: int, sex: str,
                    report_date: date, glucose: float, cholesterol: float, hemoglobin: float):
    """Render a one-page lab report PDF."""
    c = canvas.Canvas(str(out_path), pagesize=letter)
    c.setFont("Helvetica-Bold", 14); c.drawString(72, 720, "Lakeside Clinic — Lab Report")
    c.setFont("Helvetica", 10)
    c.drawString(72, 700, f"Patient: {name}")
    c.drawString(72, 685, f"Age: {age}    Sex: {sex}")
    c.drawString(72, 670, f"Report date: {report_date}")
    c.line(72, 660, 540, 660)
    c.setFont("Helvetica-Bold", 11); c.drawString(72, 640, "Test Results")
    c.setFont("Helvetica", 10)
    c.drawString(90, 620, f"Glucose:      {glucose} mg/dL")
    c.drawString(90, 605, f"Cholesterol:  {cholesterol} mg/dL")
    c.drawString(90, 590, f"Hemoglobin:   {hemoglobin} g/dL")
    c.save()


def build_lab_dataset(out_dir: Path, *, use_real_names: bool, seed: int):
    """Generate index.csv + one PDF per row in `out_dir`. Returns the directory."""
    random.seed(seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, (real_name, age, sex) in enumerate(REAL_PATIENTS, start=1):
        patient_id = f"P{i:03d}"
        # Mock uses placeholder names; private uses real-looking names
        name = real_name if use_real_names else f"Patient MOCK{i:03d}"
        report_date = date(2025, 6, 1) + timedelta(days=random.randint(0, 90))
        glucose     = round(random.gauss(95, 15), 1)
        cholesterol = round(random.gauss(190, 30), 1)
        hemoglobin  = round(random.gauss(14, 1.5), 1)
        report_filename = f"report_{patient_id}.pdf"

        generate_lab_pdf(out_dir / report_filename, name, age, sex,
                        report_date, glucose, cholesterol, hemoglobin)
        rows.append({
            "patient_id":         patient_id,
            "patient_name":       name,
            "age":                age,
            "sex":                sex,
            "report_date":        report_date.isoformat(),
            "glucose_mg_dl":      glucose,
            "cholesterol_mg_dl":  cholesterol,
            "hemoglobin_g_dl":    hemoglobin,
            "report_filename":    report_filename,
        })

    with open(out_dir / "index.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    return out_dir


lab_mock_dir    = DATA_DIR / "lab_mock"
lab_private_dir = DATA_DIR / "lab_private"
build_lab_dataset(lab_mock_dir,    use_real_names=False, seed=1)
build_lab_dataset(lab_private_dir, use_real_names=True,  seed=42)

print(f"✅  Mock    : {lab_mock_dir} ({len(list(lab_mock_dir.iterdir()))} files)")
print(f"✅  Private : {lab_private_dir} ({len(list(lab_private_dir.iterdir()))} files)")


✅  Mock    : /Users/falcon.ronnie/data/lab_mock (11 files)
✅  Private : /Users/falcon.ronnie/data/lab_private (11 files)


Optional: write a README that's shared alongside the mock data. The DS sees this when they browse the dataset.

In [54]:
lab_readme = DATA_DIR / "lab_readme.md"
lab_readme.write_text("""# Lab Reports Dataset

Each row in `index.csv` corresponds to a patient lab report. The PDF for that
record is in the same folder, named per the `report_filename` column.

## Columns

- `patient_id` — primary key, format `P###`
- `patient_name` — full name (mock data uses placeholder names)
- `age`, `sex` — demographics
- `report_date` — ISO date of the lab visit
- `glucose_mg_dl`, `cholesterol_mg_dl`, `hemoglobin_g_dl` — test values
- `report_filename` — companion PDF in the same folder

## Suggested DS analyses

- Aggregate statistics on test values
- Extract text from PDFs (e.g. with `pypdf`) and look for patterns
""")
print(f"✅  README written to {lab_readme}")


✅  README written to /Users/falcon.ronnie/data/lab_readme.md


In [56]:
dataset = client.create_dataset(
    name="lab_reports1",
    mock_path=lab_mock_dir,         # folder → all files inside become mock_files
    private_path=lab_private_dir,
    summary="Patient lab reports — CSV index with companion PDFs",
    readme_path=lab_readme,
    tags=["medical", "pdf", "tabular"],
    users=DS_EMAILS,
)
print(f"✅  Dataset {dataset.name!r} created with {len(list(lab_mock_dir.iterdir()))} mock files")


✅  Dataset 'lab_reports1' created with 11 mock files


### Verify what's been published

`client.datasets.get_all()` lists every dataset on this datasite (yours + any you've published).

In [57]:
for ds in client.datasets.get_all():
    print(f"  • {ds.name}  —  {ds.summary or '(no summary)'}")


  • beach_water_quality  —  Beach water quality readings (90 days, 5 stations)
  • beach_water_quality1  —  Beach water quality readings (90 days, 5 stations)
  • lab_reports  —  Patient lab reports — CSV index with companion PDFs
  • lab_reports1  —  Patient lab reports — CSV index with companion PDFs


---
## Step 7 — List submitted jobs

Once a Data Scientist submits a job to you, it appears in `client.jobs`. The status reflects where it is in the lifecycle:

| Status | Meaning |
|--------|---------|
| 📥 `received` | Files have arrived but the runner hasn't validated them yet. |
| ⏳ `pending` | Validated and **waiting for your review**. |
| 🔄 `approved` | You approved it. Will run on the next call to `client.process_approved_jobs()`. |
| ⚙️ `running` | Currently executing on your machine. |
| ✅ `done` | Finished successfully. |
| 🚫 `rejected` | You rejected it. |
| ❌ `failed` | The DS's script crashed during execution. |

In [58]:
emoji = {
    "received": "📥", "pending":  "⏳", "approved": "🔄",
    "running":  "⚙️",  "done":     "✅", "rejected": "🚫",
    "failed":   "❌",
}

if not client.jobs:
    print("📭 No jobs submitted yet.")
else:
    print(f"📋 Jobs ({len(client.jobs)} total):\n")
    for i, job in enumerate(client.jobs):
        icon = emoji.get(job.status, "❓")
        print(f"  [{i}]  {icon} {job.status:9s}  {job.name}  (from {job.submitted_by})")


📋 Jobs (3 total):

  [0]  ⏳ pending    Beach Quick Stats · 2026-05-07 22:29:52  (from falcon.ronnie@gmail.com)
  [1]  ⏳ pending    Beach Statistical Analysis · 2026-05-07 22:29:44  (from falcon.ronnie@gmail.com)
  [2]  ⏳ pending    Beach Simple Analysis · 2026-05-07 22:29:29  (from falcon.ronnie@gmail.com)


---
## Step 8 — Review a job

Before approving, **read the source code** the Data Scientist submitted. This is the moment where you decide whether the script is safe to run on your private data.

Things to look for:
- Does the script only use `resolve_dataset_file_path(...)` to read data, or does it try to access other files on your system?
- Does it write only to `outputs/`, or to other locations?
- Are the dependencies (`requirements.txt` if present) ones you trust?

In [60]:
from IPython.display import Code, Markdown, display

JOB_INDEX = 0   # ✏️ index from the list above

if JOB_INDEX < 0 or JOB_INDEX >= len(client.jobs):
    print(f"❌ JOB_INDEX={JOB_INDEX} is out of range. Pick 0..{len(client.jobs) - 1}.")
else:
    job = client.jobs[JOB_INDEX]
    display(Markdown(
        f"### 📄 `[{JOB_INDEX}]` {job.name}\n"
        f"- **Status:** `{job.status}`\n"
        f"- **Submitted:** {job.submitted_at}\n"
        f"- **Submitter:** {job.submitted_by}\n\n"
        f"**Code submitted** (review carefully before approving):"
    ))
    display(Code(job.code, language="python"))


### 📄 `[0]` Beach Quick Stats · 2026-05-07 22:29:52
- **Status:** `pending`
- **Submitted:** 2026-05-07T22:29:54.814409+00:00
- **Submitter:** falcon.ronnie@gmail.com

**Code submitted** (review carefully before approving):

"""Auto-approval job: reads params.json, computes the requested metric."""
import csv
import json
import os
from syft_client import resolve_dataset_file_path

with open("params.json") as f:
    params = json.load(f)

data_path = resolve_dataset_file_path(params["dataset"])

with open(data_path, newline="") as f:
    rows = list(csv.DictReader(f))

total = len(rows)
safe  = sum(1 for r in rows if r["safe_to_swim"].strip().lower() == "true")

result = {
    "dataset":        params["dataset"],
    "metric":         params["metric"],
    "description":    params["description"],
    "total_readings": total,
    "pct_safe":       round(safe / total * 100, 1),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

### Approve or reject

If the code looks safe, call `job.approve()`. To reject, use `job.reject(reason="...")` — the reason is shared back to the DS so they can resubmit a corrected version.

In [30]:
# Approve the job picked above (uncomment one of the lines below):

job.approve()
# job.reject(reason="Script reads files outside the dataset; please scope access.")

print(f"Job is currently: {job.status!r}")


NameError: name 'job' is not defined

---
## Step 9 — Run approved jobs

Approving a job doesn't run it — `client.process_approved_jobs()` does. Each approved job runs in its own isolated Python virtual environment (the DS's `dependencies=[...]` are installed automatically).

### Sharing results with the submitter

By default, results stay on your machine. To send them back to the DS, pass `share_outputs_with_submitter=True`. Almost always what you want.

> 💡 Alternative: leave it `False` for jobs where you want to inspect the outputs first, then call `job.share_outputs([ds_email])` manually after review.

In [31]:
client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,    # share stdout/stderr too — helps the DS debug
)


---
## Step 10 — Run unattended with `syft_bg`

Up to now you've been doing each step manually. For a real deployment you'll want background services that run without you in the loop. `syft-bg` provides four:

| Service | What it does |
|---------|--------------|
| 🔄 **`sync`** | Keeps your SyftBox in sync with Drive — new submissions appear without manual `client.sync()` calls. |
| 🔔 **`notify`** | Sends you an email when a new job arrives or a peer requests connection. |
| 🤖 **`approve`** | Applies your auto-approval rules (Step 12) — and, optionally, auto-approves peer requests from whitelisted domains. |
| 📧 **`email_approve`** | Lets you approve/reject jobs by replying to the notification emails. Requires a Google Cloud Pub/Sub topic. |

### Three-stage setup

1. **`syft_bg.init(...)`** writes the configuration. One-time — re-run it only when you want to change settings.
2. **`syft_bg.ensure_running(services=[...])`** starts the daemons. Re-run any time (e.g. after a reboot) — it's idempotent.
3. **`syft_bg.status`** shows which services are alive. Use this as your sanity check.

> ⚠️ Background services are **detached processes**: they survive the Jupyter kernel restarting and they survive you closing this notebook tab. Use `syft_bg.reset()` (Step 13) to fully clear and restart.


In [32]:
import syft_bg

# ✏️ Required ONLY if you want the email_approve service.
# Create a Pub/Sub topic in your GCP project — see the syft-bg docs for the setup.
GCP_PROJECT_ID = ""   # ✏️ e.g. "my-syftbox-project" — leave empty to skip email_approve

settings = {}
if GCP_PROJECT_ID:
    settings["email_approve"] = {"gcp_project_id": GCP_PROJECT_ID}

syft_bg.init(
    do_email=DO_EMAIL,
    token_path=str(TOKEN_PATH),
    settings=settings,
)


Stored token at /Users/falcon.ronnie/.syft-bg/token.json
Config saved to /Users/falcon.ronnie/.syft-bg/config.yaml


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.8) which Google will stop supporting in new releases of google.pubsub_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.pubsub_v1 past that date.
  warnings.warn(message, FutureWarning)


Now start the services. Pass the list of services to run — `restart=True` makes the call idempotent (it'll restart anything already running with the new config).

> 💡 If you skipped `email_approve` (no GCP project ID), drop it from the list below.


In [33]:
services = ["sync", "notify", "approve"]
if GCP_PROJECT_ID:
    services.append("email_approve")

syft_bg.ensure_running(services=services, restart=True)


sync started
notify started
approve started


### Sanity check: are they actually running?

`syft_bg.status` prints a snapshot of every service. After `ensure_running()`, you want to see each service marked **🟢 running** with a recent log timestamp.

In [34]:
syft_bg.status


syft-bg status
========================================
  email:       ronnie@openmined.org
  syftbox:     /Users/falcon.ronnie/SyftBox_ronnie@openmined.org
  environment: local

tokens
----------------------------------------
  notify.gmail_token_path             ready
  notify.drive_token_path             ready
  approve.drive_token_path            ready
  email_approve.gmail_token_path      ready

services
----------------------------------------
  notify           🟡 starting (PID 46224) (not installed)
  approve          🟡 starting (PID 46225) (not installed)
  email_approve    ⚪ stopped (not installed)
  sync             🟡 starting (PID 46223) (not installed)

---
## Step 10 — Auto-approval for trusted job patterns

Reviewing every job by hand gets old fast for repeat workflows. **Auto-approval** lets you whitelist a specific `main.py` (matched by content hash) so future jobs with the same code run without manual review. The DS can change `params.json` between runs without invalidating the rule.

> 💡 Auto-approval rules need the `approve` service from Step 10 to actually do anything — without it, your rules are just sitting in a config file.

### How matching works

| Match type | When to use it |
|------------|----------------|
| **By content (hash)** | The file shouldn't change. Used for `main.py` — if the DS modifies the script, you want to re-review. |
| **By name only** | The file's name is fixed but its contents change between runs. Used for `params.json`. |

### The cleanest pattern

Take a *pending* job you've already reviewed and turn it directly into a rule:

```python
syft_bg.auto_approve_job(
    job,
    file_paths=["params.json"],   # name-matched (the DS can edit params freely)
    # all other files default to content-matched
    # peers=[ds_email]            # optional: restrict to a specific DS
)
```

After registering, the `approve` service picks up the rule and approves matching jobs automatically — no notebook needed.


In [36]:
import syft_bg

# Pick a pending job to use as the auto-approval template
JOB_INDEX_FOR_AUTO = 0   # ✏️ index from the Step 7 list

candidate = client.jobs[JOB_INDEX_FOR_AUTO]
print(f"Using as template: {candidate.name!r} (status: {candidate.status})")
print(f"Files in submission:")
for f in candidate.files:
    print(f"   • {f.name}")


Using as template: 'Beach Quick Stats · 2026-05-07 22:29:52' (status: pending)
Files in submission:
   • config.yaml
   • run.sh
   • params.json
   • main.py
   • state.yaml


In [ ]:
# Register the rule. main.py is content-hashed; params.json matches by name only.
# Adjust file_paths if the job uses different file names.

result = syft_bg.auto_approve_job(
    candidate,
    file_paths=["params.json"],   # ✏️ name-matched files (parameters that can change)
)
if result.success:
    print(f"✅  Rule registered as {result.name!r}")
else:
    print(f"❌  Failed: {result.error}")


---
## Step 12 — Inspect service logs

If something looks off — jobs not getting picked up, emails not arriving — `syft_bg.logs(service, n=20)` shows the most recent log lines for that service.

### Healthy idle output

When the services are running but there's nothing to do, you'll see lines like `"No new jobs"` or `"No matching auto-approval rules"`. That's normal — it means the service is alive and polling.


In [ ]:
SERVICE = "approve"   # ✏️ one of: "notify", "approve"
N_LINES = 20

lines = syft_bg.logs(SERVICE, n=N_LINES)
if not lines:
    print(f"(no log entries yet for {SERVICE!r} — give it a minute, then re-run)")
else:
    for line in lines:
        print(line)


---
## Step 13 — Reset (only if you need to start over)

For day-to-day operation, you don't need to do anything — leave the background services running. They'll quietly handle new jobs and peer events.

If something goes wrong and you want to start from a clean slate, **`syft_bg.reset()`** stops every service AND clears their installed state, so the next `init` + `ensure_running` brings up a fresh setup.

> ⚠️ This **doesn't** delete your datasets, peers, or auto-approval rules — those live in your SyftBox folder on Drive, not in `syft-bg`'s state.


In [ ]:
# Uncomment ONLY if you want to wipe syft-bg state and start fresh:

# syft_bg.reset()
# print("✅ All services stopped and uninstalled. Re-run Step 11 to start again.")


In [ ]:
# Remove a dataset you no longer want to host (uncomment to use):
# client.delete_dataset("lab_reports")
# client.delete_dataset("beach_water_quality")


---
## 🎉 You're done!

You've successfully:
- ✅ Logged in as a Data Owner
- ✅ Pre-added your Data Scientists as peers
- ✅ Created a single-file dataset and a CSV-with-PDFs dataset
- ✅ Reviewed and approved a Data Scientist's job
- ✅ Ran the job and shared the results back
- ✅ Started background services for unattended operation
- ✅ Set up an auto-approval rule for a trusted job pattern

## Day-to-day workflow

After initial setup, ongoing operation is mostly hands-off:

```
syft_bg.status                    # confirm services are running (e.g. after a reboot)
# Receive emails when new jobs arrive (notify service)
# Reply 'approve' to the email — email_approve runs the job for you
# Or, in the notebook: client.process_approved_jobs(share_outputs_with_submitter=True)
```

If something gets stuck, `syft_bg.reset()` + re-run Step 11 brings up a clean state.

## Resources

- [`syft-client` docs](https://www.mintlify.com/OpenMined/syft-client)
- [`syft-bg` background services docs](https://www.mintlify.com/OpenMined/syft-client/packages/syft-bg)
- [API reference](https://www.mintlify.com/OpenMined/syft-client/api/client/login)
- [OpenMined Community Slack](https://slack.openmined.org)
